In [1]:
import os
import gc          # garbage collector — I use this to manually free RAM throughout
import tempfile     # for creating a temp folder where memmap files go
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_XLA_FLAGS'] = '--tf_xla_cpu_global_jit=false --tf_xla_auto_jit=0'
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/home/alex/miniconda3/envs/jupyterproject-project'

import pandas as pd
import numpy as np
import optuna
import warnings
import logging
import tensorflow as tf

# Configure GPU memory growth
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

tf.config.optimizer.set_jit(False)
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.losses import Huber
from tensorflow.keras.regularizers import l2

warnings.filterwarnings('ignore', category=FutureWarning)
optuna.logging.set_verbosity(logging.WARNING)

# ==========================================
# 1. LOAD, FILTER, AND PREPARE DATA
# ==========================================
# Same hourly data loading pipeline as my per-product hourly LSTM
print("Loading and filtering data...")

data_dir = '../preprocesing_data/processed_csv'
sales_path = os.path.join(data_dir, 'sales_data_preprocessed.csv')
weather_path = os.path.join(data_dir, 'weather_data_hourly.csv')
holidays_path = os.path.join(data_dir, 'holidays_data_preprocessed.csv')
events_path = os.path.join(data_dir, 'aberdeen_events_master_timeline.csv')

# 1.1 Load Sales Data (Hourly)
sales_df = pd.read_csv(sales_path)
sales_df['Date'] = pd.to_datetime(sales_df['Date'])
hourly_sales = sales_df.groupby('Date').sum(numeric_only=True).reset_index()

# AUTO-DETECT ALL PRODUCT COLUMNS
non_product_cols = ['Date', 'Time', 'date', 'Date_Norm']
product_columns = [col for col in hourly_sales.columns 
                   if col not in non_product_cols 
                   and hourly_sales[col].dtype.kind in 'iufc']
print(f"Detected {len(product_columns)} products: {product_columns}")

# 1.2 Load Weather Data
weather_df = pd.read_csv(weather_path)
weather_df['Date'] = pd.to_datetime(weather_df['Date'])

def create_weather_flags(df):
    if 'weather_code' not in df.columns:
        return df
    df['is_clear'] = (df['weather_code'] == 0).astype(int)
    df['is_cloudy'] = df['weather_code'].isin([1, 2, 3, 45, 48]).astype(int)
    df['is_rain'] = df['weather_code'].isin([51, 53, 55, 56, 57, 61, 63, 65, 66, 67, 80, 81, 82, 95, 96, 99]).astype(int)
    df['is_snow'] = df['weather_code'].isin([71, 73, 75, 77, 85, 86]).astype(int)
    return df.drop(columns=['weather_code'])

weather_df = create_weather_flags(weather_df)
hourly_weather = weather_df

# 1.3 Load Holidays & Events
holidays_df = pd.read_csv(holidays_path)
holidays_df['Date'] = pd.to_datetime(holidays_df['Date'])
daily_holidays = holidays_df.groupby(holidays_df['Date'].dt.normalize()).max(numeric_only=True).reset_index()
daily_holidays = daily_holidays.rename(columns={'Date': 'Date_Only'})

events_df = pd.read_csv(events_path)
events_df['Date'] = pd.to_datetime(events_df['Date'])
events_df['Date_Only'] = events_df['Date'].dt.normalize()

# 1.4 Merge External Features with Hourly Sales
df = hourly_sales.copy()

# Time features (hourly cyclical encoding for the 9-hour business window)
df['Hour_of_Day'] = df['Date'].dt.hour
df['Day_of_Week'] = df['Date'].dt.dayofweek
df['Is_Weekend'] = df['Day_of_Week'].isin([5, 6]).astype(int)
df['hour_sin'] = np.sin(2 * np.pi * (df['Hour_of_Day'] - 8) / 9)
df['hour_cos'] = np.cos(2 * np.pi * (df['Hour_of_Day'] - 8) / 9)
df['day_sin'] = np.sin(2 * np.pi * df['Day_of_Week'] / 7)
df['day_cos'] = np.cos(2 * np.pi * df['Day_of_Week'] / 7)
df['month_sin'] = np.sin(2 * np.pi * (df['Date'].dt.month - 1) / 12)
df['month_cos'] = np.cos(2 * np.pi * (df['Date'].dt.month - 1) / 12)
df = df.drop(columns=['Hour_of_Day', 'Day_of_Week'])

df['Date_Only'] = df['Date'].dt.normalize()

# Merge weather (hourly)
df = df.merge(hourly_weather, on='Date', how='left')

# Merge holidays (daily)
holiday_features = [col for col in daily_holidays.columns if col not in ['Date_Only']]
df = df.merge(daily_holidays, on='Date_Only', how='left')

# Merge events (daily)
daily_events = events_df.groupby('Date_Only').sum(numeric_only=True).reset_index()
event_features = [col for col in daily_events.columns if col not in ['Date', 'Date_Only']]
df = df.merge(daily_events[['Date_Only'] + event_features], on='Date_Only', how='left')

df = df.drop(columns=['Date_Only'])

# Fill gaps
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].fillna(0)

flag_cols = [col for col in df.columns if col.startswith('is_')]
if flag_cols:
    df[flag_cols] = df[flag_cols].clip(upper=1)

# Base features (shared across all products)
time_features = ['hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'Is_Weekend', 'month_sin', 'month_cos']
weather_features = [col for col in hourly_weather.columns if col not in ['Date']]

base_feature_cols = time_features + weather_features + holiday_features + event_features
base_feature_cols = [col for col in base_feature_cols 
                     if col in df.columns and df[col].dtype.kind in 'iufc']

print(f"Base features (shared): {len(base_feature_cols)}")
# Downcast all float64 columns to float32 before the melt in the next cell.
# float32 gives ~7 decimal digits of precision which is more than enough
# for scaled 0-1 values, and it halves the memory of every array downstream.
for col in df.select_dtypes(include=['float64']).columns:
    df[col] = df[col].astype(np.float32)

# These source dataframes have all been merged into df now,
# so I can free them. Keeping df because clustering (Cell 2) still needs it.
del sales_df, hourly_sales, weather_df, hourly_weather, holidays_df, events_df, daily_holidays, daily_events
gc.collect()

print(f"Products to forecast: {product_columns}")

# ==========================================
# PRODUCTS TO FORECAST
# ==========================================
# These are the 74 specific menu items I want to predict
PRODUCTS_TO_FORECAST = [
 'Avo & Hal Muffin',
 'Avo, Egg & Bacon',
 'Avo, Feta & Tom',
 'Avocado on Toast',
 'Bacon',
 'Bacon Bap',
 'Bacon Egg Brioch',
 'Bacon Waffle',
 'Baked Beans',
 'Baked Beans JP',
 'Bean Soldiers',
 'Big Breakfast',
 'Black Pudding',
 'Breakfast Hash',
 'Breakfast Muffin',
 'Breakfast Wrap',
 'Buttd Mushrooms',
 'Cheese & Bean JP',
 'Cheese JP',
 'Chick Flatbread',
 'Chicken Club',
 'Chilli Carne JP',
 'Egg Bacon Brioch',
 'Egg Bap',
 'Extra Beans',
 'F.Eggs on Toast',
 'Festive Stack',
 'Fried Egg',
 'Hash Brown',
 'Hash Brown Bites',
 'Little Avo Toast',
 'Little Bean Toas',
 'Little Egg Toast',
 'Ltle Bfast Bacon',
 'Ltle Bfast Saus',
 'Mini Hash Browns',
 'P.Eggs on Toast',
 'Poached Egg',
 'Posh Beans',
 'Roll & Butter ',
 'S.Eggs on Toast',
 'Sausage',
 'Sausage Bap',
 'Scrambled Egg',
 'Streaky Bacon',
 'Tattie Scone',
 'The Breakfast',
 'Toasted Teacake',
 'Tuna JP',
 'Tuna Mayo Mix',
 'Tuna Melt Panini',
 'Tuna Panini',
 'Tuna Toastie',
 'Veg Sausage Bap',
 'Vegan Breakfast',
 'Vegan Sausage',
 'Veggie Bap',
 'Veggie Breakfast',
 'Bakery',
 'White Toast Bread',
 'Brown Toast Bread',
 'Porridge',
 'Sourdough Toast Bread',
 'Multiseed Toast Bread',
]


/home/alex/miniconda3/envs/jupyterproject-project/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading and filtering data...
Detected 222 products: ['Almd & Aprct Bar', 'Apl & Blk Smthie', 'Avo & Hal Muffin', 'Avo, Egg & Bacon', 'Avo, Feta & Tom', 'Avocado on Toast', 'BD Curry Roll', 'BLT', 'Bacon', 'Bacon Bap', 'Bacon Egg Brioch', 'Bacon Waffle', 'Baked Beans', 'Baked Beans JP', 'Banana Bread', 'Bean Soldiers', 'Beef Burger', 'Beef Hors Foca', 'Beyond Burger', 'Big Breakfast', 'Black Pudding', 'Blk Forest Syrup', 'Bre Cran Toastie', 'Breakfast Bap', 'Breakfast Hash', 'Breakfast Muffin', 'Breakfast Wrap', 'Brie Bacon Crois', 'Brie Cranb Tstie', 'Brunch Burger', 'Buttd Mushrooms', 'Butter', 'Cheese & Bean JP', 'Cheese JP', 'Cheese Pizza', 'Cheese Sand PnM', 'Cheese Sandwich', 'Cheese Scone', 'Chick Chzo Tstie', 'Chick Flatbread', 'Chick Rice Bowl', 'Chicken Breast', 'Chicken Burger', 'Chicken Club', 'Chicken Goujons', 'Chicken Strips', 'Chicken Waffles', 'Chicken Wrap', 'Chickn Rice Bowl', 'Chilli Carne JP', 'Chips', 'Chorizo', 'Chorizo & Eggs', 'Chse Onion Tstie', 'Chse Tom Toas

In [2]:
# ==========================================
# 2. MELT TO LONG FORMAT & BUILD GLOBAL HOURLY FEATURES
# ==========================================
# Same approach as my global daily LSTM but with hourly lags:
# 1h ago, 9h ago (~1 business day), 63h ago (~1 business week)

SEQUENCE_LENGTH = 63  # ~1 week of hourly data (same as per-product version)

train_end = pd.to_datetime('2025-10-01')
val_end = pd.to_datetime('2025-11-01')
test_end = pd.to_datetime('2025-11-30')

print("Melting to long format (one row per product per hour)...")

# Melt: wide -> long
df_long = pd.melt(df, id_vars=['Date'] + base_feature_cols, value_vars=product_columns,
                  var_name='Product_Name', value_name='Sales')
df_long['Sales'] = df_long['Sales'].clip(lower=0)
df_long = df_long.sort_values(['Product_Name', 'Date']).reset_index(drop=True)

# Encode product name as an integer
product_encoder = LabelEncoder()
df_long['Product_ID'] = product_encoder.fit_transform(df_long['Product_Name'])
all_product_names = product_encoder.classes_
n_products = len(all_product_names)
print(f"Encoded {n_products} products as integers 0-{n_products-1}")

# Add hourly lag features grouped by product (same lags as per-product version)
df_long['sales_lag_1h'] = df_long.groupby('Product_Name')['Sales'].shift(1)
df_long['sales_lag_9h'] = df_long.groupby('Product_Name')['Sales'].shift(9)
df_long['sales_lag_63h'] = df_long.groupby('Product_Name')['Sales'].shift(63)
df_long['sales_rolling_9h_mean'] = df_long.groupby('Product_Name')['Sales'].shift(1).groupby(
    df_long['Product_Name']).transform(lambda x: x.rolling(9, min_periods=1).mean())
df_long['sales_rolling_9h_std'] = df_long.groupby('Product_Name')['Sales'].shift(1).groupby(
    df_long['Product_Name']).transform(lambda x: x.rolling(9, min_periods=1).std()).fillna(0)

lag_features = ['sales_lag_1h', 'sales_lag_9h', 'sales_lag_63h', 'sales_rolling_9h_mean', 'sales_rolling_9h_std']

# Drop rows where lags aren't available yet
df_long = df_long.dropna(subset=lag_features).reset_index(drop=True)

# Feature columns: base + product ID + hourly lags
feature_cols = base_feature_cols + ['Product_ID'] + lag_features

print(f"Long format shape: {df_long.shape}")
print(f"Features ({len(feature_cols)}): {feature_cols}")
# Same float32 downcast on df_long — this is the biggest dataframe in the
# pipeline (one row per product per hour), so halving its memory matters a lot.
for col in df_long.select_dtypes(include=['float64']).columns:
    df_long[col] = df_long[col].astype(np.float32)
print(f"df_long memory: {df_long.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"Date range: {df_long['Date'].min()} to {df_long['Date'].max()}")

Melting to long format (one row per product per hour)...
Encoded 222 products as integers 0-221
Long format shape: (2905092, 36)
Features (33): ['hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'Is_Weekend', 'month_sin', 'month_cos', 'apparent_temperature', 'precipitation', 'snowfall', 'snow_depth', 'relative_humidity_2m', 'cloud_cover', 'visibility', 'wind_speed_10m', 'wind_gusts_10m', 'is_clear', 'is_cloudy', 'is_rain', 'is_snow', 'is_holiday', 'holiday_importance', 'Days_Until_Next_Holiday', 'is_festival', 'festival_importance', 'is_match_day', 'match_importance', 'Product_ID', 'sales_lag_1h', 'sales_lag_9h', 'sales_lag_63h', 'sales_rolling_9h_mean', 'sales_rolling_9h_std']
df_long memory: 749.4 MB
Date range: 2022-01-08 08:00:00 to 2025-12-31 16:00:00


In [3]:
# ==========================================
# 2B. DATA-DRIVEN PRODUCT CLUSTERING (FOR OPTUNA ONLY)
# ==========================================
#
# Same as daily: clusters are used ONLY to speed up Optuna.
# The final model is still ONE global hourly LSTM.

from sklearn.preprocessing import StandardScaler as ClusterScaler
from sklearn.cluster import KMeans

print("Building product clusters (for Optuna speedup only)...")

# Aggregate hourly to daily for clustering features
train_hourly_cl = df_long[df_long['Date'] <= train_end].copy()
train_hourly_cl['Date_Only'] = train_hourly_cl['Date'].dt.normalize()
train_daily_agg = train_hourly_cl.groupby(['Product_Name', 'Date_Only'])['Sales'].sum().reset_index()
train_daily_agg['dow'] = pd.to_datetime(train_daily_agg['Date_Only']).dt.dayofweek

if 'apparent_temperature' in df.columns:
    temp_daily = df[['Date', 'apparent_temperature']].copy()
    temp_daily['Date_Only'] = temp_daily['Date'].dt.normalize()
    temp_daily = temp_daily.groupby('Date_Only')['apparent_temperature'].mean().reset_index()
else:
    temp_daily = None

product_features = []
for product in PRODUCTS_TO_FORECAST:
    prod_daily = train_daily_agg[train_daily_agg['Product_Name'] == product]
    if len(prod_daily) == 0:
        continue
    series = prod_daily['Sales'].values.astype(float)
    mean_vol = np.mean(series)
    cv = np.std(series) / mean_vol if mean_vol > 0 else 999.0
    zero_frac = np.mean(series == 0)
    weekend_mask = prod_daily['dow'].isin([5, 6])
    wkend_mean = prod_daily.loc[weekend_mask, 'Sales'].mean() if weekend_mask.sum() > 0 else 0
    wkday_mean = prod_daily.loc[~weekend_mask, 'Sales'].mean() if (~weekend_mask).sum() > 0 else 0
    weekend_ratio = wkend_mean / wkday_mean if wkday_mean > 0 else 1.0
    temp_corr = 0.0
    if temp_daily is not None:
        merged = prod_daily.merge(temp_daily, on='Date_Only', how='inner')
        if len(merged) > 10:
            tc = merged[['Sales', 'apparent_temperature']].corr().iloc[0, 1]
            temp_corr = tc if not np.isnan(tc) else 0.0
    product_features.append({
        'product': product, 'mean_daily_volume': mean_vol,
        'coeff_of_variation': cv, 'zero_day_fraction': zero_frac,
        'weekend_ratio': weekend_ratio, 'temp_correlation': temp_corr
    })

cluster_features_df = pd.DataFrame(product_features)

N_CLUSTERS = 4
X_cluster = cluster_features_df[['mean_daily_volume', 'coeff_of_variation',
    'zero_day_fraction', 'weekend_ratio', 'temp_correlation']].values
X_cluster_scaled = ClusterScaler().fit_transform(X_cluster)
cluster_features_df['cluster_id'] = KMeans(
    n_clusters=N_CLUSTERS, random_state=42, n_init=10
).fit_predict(X_cluster_scaled)

CLUSTER_PRODUCTS = {}
CLUSTER_HEROES = {}
for cid in range(N_CLUSTERS):
    cp = cluster_features_df[cluster_features_df['cluster_id'] == cid]
    CLUSTER_HEROES[cid] = cp.sort_values('mean_daily_volume', ascending=False).iloc[0]['product']
    CLUSTER_PRODUCTS[cid] = cp['product'].tolist()

for cid in range(N_CLUSTERS):
    print(f"  Cluster {cid} ({len(CLUSTER_PRODUCTS[cid])} products), "
          f"hero={CLUSTER_HEROES[cid]}: {', '.join(CLUSTER_PRODUCTS[cid][:6])}...")
print(f"\nClusters built. Used ONLY for fast Optuna — final model is global.")

del df
gc.collect()


Building product clusters (for Optuna speedup only)...
  Cluster 0 (28 products), hero=Bacon Bap: Avocado on Toast, Bacon, Bacon Bap, Bacon Waffle, Baked Beans, Big Breakfast...
  Cluster 1 (30 products), hero=Breakfast Muffin: Avo & Hal Muffin, Avo, Egg & Bacon, Avo, Feta & Tom, Bacon Egg Brioch, Baked Beans JP, Bean Soldiers...
  Cluster 2 (3 products), hero=Festive Stack: Festive Stack, Tuna Mayo Mix, Tuna Melt Panini...
  Cluster 3 (3 products), hero=Bakery: The Breakfast, Bakery, White Toast Bread...

Clusters built. Used ONLY for fast Optuna — final model is global.


0

In [4]:
# ==========================================
# 3. BUILD SEQUENCES FOR ALL PRODUCTS AT ONCE
# ==========================================
# MEMORY OPTIMISATION NOTES:
# The original version stored all sequences in RAM as numpy arrays.
# With 74 products × thousands of sequences × 63 timesteps × 30 features,
# that was easily 5-10+ GB and crashed my machine.
#
# Fix: I use np.memmap to store the sequence arrays as files on disk.
# The OS pages in only what's needed, so RAM stays low.
# The model sees exactly the same data — just delivered from disk not RAM.
#
# I also pre-compute the naive MAE (needed for MASE metric later)
# and save df_long to parquet, so I can free it from RAM entirely.
# Cell 6 reloads it from disk when it needs the lag history for forecasting.
print("Building hourly sequences for all products...")

# Split data by time
train_long = df_long[df_long['Date'] <= train_end].copy()
print(f"Train rows: {len(train_long)}")

# Fit scalers on training data only
feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()
feature_scaler.fit(train_long[feature_cols])
target_scaler.fit(train_long[['Sales']])
# train_long was only needed to fit the scalers, done now
del train_long
gc.collect()

# --- Pre-compute in-sample naive MAE per product (needed for MASE later) ---
# Normally Cells 6 and 8 would re-scan df_long for every product to get this.
# By computing it now, I can delete df_long from RAM after this cell.
print("  Pre-computing naive MAE per product...")
product_naive_mae = {}
for product_name in product_columns:
    product_hourly_train = df_long[
        (df_long['Product_Name'] == product_name) & (df_long['Date'] <= train_end)
    ].copy()
    product_hourly_train['Date_Only'] = product_hourly_train['Date'].dt.date
    product_daily_train = product_hourly_train.groupby('Date_Only')['Sales'].sum().reset_index()
    if len(product_daily_train) > 1:
        product_naive_mae[product_name] = np.mean(np.abs(np.diff(product_daily_train['Sales'].values)))
    else:
        product_naive_mae[product_name] = 1.0
    del product_hourly_train, product_daily_train
gc.collect()

def build_product_sequences(product_name, seq_len):
    """Build hourly sequences for a single product using global scalers."""
    product_data = df_long[df_long['Product_Name'] == product_name].sort_values('Date').reset_index(drop=True)
    
    if len(product_data) < seq_len + 50:
        return None
    
    scaled_features = feature_scaler.transform(product_data[feature_cols]).astype(np.float32)
    scaled_targets = target_scaler.transform(product_data[['Sales']]).flatten().astype(np.float32)
    dates = product_data['Date'].values
    
    n_seqs = len(scaled_features) - seq_len
    n_feat = scaled_features.shape[1]
    
    X_seqs = np.empty((n_seqs, seq_len, n_feat), dtype=np.float32)
    y_seqs = np.empty(n_seqs, dtype=np.float32)
    
    for i in range(n_seqs):
        X_seqs[i] = scaled_features[i : i + seq_len]
        y_seqs[i] = scaled_targets[i + seq_len]
    
    date_seqs = pd.to_datetime(dates[seq_len:])
    
    train_mask = date_seqs <= train_end
    val_mask = (date_seqs > train_end) & (date_seqs <= val_end)
    test_mask = (date_seqs > val_end) & (date_seqs <= test_end)
    
    return {
        'X_train': X_seqs[train_mask], 'y_train': y_seqs[train_mask],
        'X_val': X_seqs[val_mask], 'y_val': y_seqs[val_mask],
        'X_test': X_seqs[test_mask], 'y_test': y_seqs[test_mask],
        'test_dates': date_seqs[test_mask],
        'product_name': product_name
    }

# --- Pass 1: Count sequences and collect test data per product ---
print("  Pass 1: counting sequences...")
product_seq_counts = {}
product_test_data = {}
products_included = []

for product_name in product_columns:
    result = build_product_sequences(product_name, SEQUENCE_LENGTH)
    if result is None:
        print(f"  SKIPPING {product_name}: not enough data")
        continue
    
    product_seq_counts[product_name] = {
        'n_train': len(result['X_train']),
        'n_val': len(result['X_val']),
    }
    # Store test data (small — only ~1 month per product)
    product_test_data[product_name] = {
        'X_test': result['X_test'],
        'y_test': result['y_test'],
        'test_dates': result['test_dates']
    }
    products_included.append(product_name)
    del result
    gc.collect()

total_train = sum(v['n_train'] for v in product_seq_counts.values())
total_val = sum(v['n_val'] for v in product_seq_counts.values())
n_feat = len(feature_cols)

mem_gb = total_train * SEQUENCE_LENGTH * n_feat * 4 / 1e9
print(f"  Total: {total_train:,} train, {total_val:,} val sequences")
print(f"  X_train would be {mem_gb:.2f} GB in RAM — using memmap instead")

# np.memmap creates a file on disk that looks like a numpy array.
# Only the pages the OS actually reads get loaded into RAM.
# So instead of 5+ GB resident, it's just whatever the OS decides to cache.

# --- Pass 2: Write sequences into the memmap files ---
# I fill these the same way as the original np.vstack approach,
# but writing into a file-backed array instead of a RAM array.
MEMMAP_DIR = tempfile.mkdtemp(prefix='lstm_memmap_')
print(f"  Memmap dir: {MEMMAP_DIR}")

X_train_path = os.path.join(MEMMAP_DIR, 'X_train.dat')
y_train_path = os.path.join(MEMMAP_DIR, 'y_train.dat')
X_val_path = os.path.join(MEMMAP_DIR, 'X_val.dat')
y_val_path = os.path.join(MEMMAP_DIR, 'y_val.dat')

X_train_global = np.memmap(X_train_path, dtype=np.float32, mode='w+',
                           shape=(total_train, SEQUENCE_LENGTH, n_feat))
y_train_global = np.memmap(y_train_path, dtype=np.float32, mode='w+',
                           shape=(total_train,))
X_val_global = np.memmap(X_val_path, dtype=np.float32, mode='w+',
                         shape=(total_val, SEQUENCE_LENGTH, n_feat))
y_val_global = np.memmap(y_val_path, dtype=np.float32, mode='w+',
                         shape=(total_val,))

train_offset = 0
val_offset = 0

for product_name in products_included:
    result = build_product_sequences(product_name, SEQUENCE_LENGTH)
    nt = product_seq_counts[product_name]['n_train']
    nv = product_seq_counts[product_name]['n_val']
    
    X_train_global[train_offset:train_offset + nt] = result['X_train']
    y_train_global[train_offset:train_offset + nt] = result['y_train']
    X_val_global[val_offset:val_offset + nv] = result['X_val']
    y_val_global[val_offset:val_offset + nv] = result['y_val']
    
    train_offset += nt
    val_offset += nv
    del result
    gc.collect()

# Make sure everything is written to disk
X_train_global.flush()
y_train_global.flush()
X_val_global.flush()
y_val_global.flush()

del product_seq_counts
gc.collect()

# --- Save df_long to parquet and free from RAM ---
# Cell 6 needs df_long to seed the lag history for recursive forecasting,
# so I save it to disk here and reload it there. Parquet is fast and compact.
DFLONG_PATH = os.path.join(MEMMAP_DIR, 'df_long.parquet')
df_long.to_parquet(DFLONG_PATH, index=False, engine='fastparquet')
print(f"  Saved df_long to {DFLONG_PATH}")
del df_long
gc.collect()

print(f"\nGlobal training set: X=({total_train}, {SEQUENCE_LENGTH}, {n_feat}), y=({total_train},)")
print(f"Global validation set: X=({total_val}, {SEQUENCE_LENGTH}, {n_feat}), y=({total_val},)")
print(f"Products included: {len(products_included)}")
print(f"Test products with data: {len(product_test_data)}")
print(f"Arrays are memory-mapped — only pages accessed are loaded into RAM")
print(f"Peak RAM should now be < 1 GB instead of the original 5-10+ GB")


Building hourly sequences for all products...
Train rows: 2721276
  Pre-computing naive MAE per product...
  Pass 1: counting sequences...
  Total: 2,707,290 train, 61,938 val sequences
  X_train would be 22.51 GB in RAM — using memmap instead
  Memmap dir: /tmp/lstm_memmap_0om535cs
  Saved df_long to /tmp/lstm_memmap_0om535cs/df_long.parquet

Global training set: X=(2707290, 63, 33), y=(2707290,)
Global validation set: X=(61938, 63, 33), y=(61938,)
Products included: 222
Test products with data: 222
Arrays are memory-mapped — only pages accessed are loaded into RAM
Peak RAM should now be < 1 GB instead of the original 5-10+ GB


In [5]:
# Helper: create a tf.data.Dataset that streams from memmap in batches.
# If I passed the memmap arrays directly to model.fit(), Keras would try to
# load them fully into RAM — defeating the whole point of memmap.
# This generator feeds one batch at a time, so only ~64 sequences are in RAM.
def make_tf_dataset(X_mm, y_mm, batch_size, shuffle=False):
    """Create a tf.data.Dataset from memmap arrays, streaming in chunks.
    Returns (dataset, steps_per_epoch) so model.fit knows exactly when
    to stop — avoids the 'input ran out of data' warning."""
    n = len(X_mm)
    steps = int(np.ceil(n / batch_size))
    def gen():
        indices = np.arange(n)
        if shuffle:
            np.random.shuffle(indices)
        for i in indices:
            yield X_mm[i], y_mm[i]
    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec(shape=(X_mm.shape[1], X_mm.shape[2]), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.float32),
        )
    )
    # .repeat() so Keras never hits end-of-data mid-epoch
    # steps_per_epoch tells it when one epoch actually ends
    ds = ds.batch(batch_size).repeat().prefetch(tf.data.AUTOTUNE)
    return ds, steps

# ==========================================
# 4. SKIPPING OPTUNA — USING BEST PARAMS FROM DAILY MODEL
# ==========================================
#
# I ran full Optuna tuning on the daily version (lstm_global_daily.ipynb)
# and it found solid params. The hourly model is architecturally identical
# so I'm transferring them directly here instead of re-running Optuna,
# which was taking way too long on hourly sequences (63 steps x 74 products).
#
# Daily winner: Cluster 0 (Bacon Waffle), Global Val RMSE: 3.2392
# If I ever want to re-tune specifically for hourly data, I can uncomment
# the original Optuna block below and comment this section out again.

# df_long is still needed later in Cell 6 for the recursive forecast seeding,
# but we no longer need to reload it here just for Optuna hero sequences.
# Cell 6 will reload it from disk as before.

best_params = {
    'batch_size':    32,
    'lstm_1_units':  160,
    'lstm_2_units':  96,
    'dropout_rate':  0.2618488090568859,
    'learning_rate': 0.00046818749855735817,
    'l2_reg':        1.005276167736066e-05,
    'dense_units':   64,   # not tuned in daily (hardcoded 64 there), keeping same default here
}
best_cluster_id = 'daily_transfer'
best_global_rmse = float('nan')

print("Skipping Optuna — params transferred from daily model (Cluster 0 / Bacon Waffle winner):")
for k, v in best_params.items():
    print(f"  {k}: {v}")

# ==========================================
# ORIGINAL OPTUNA CODE — commented out
# Uncomment everything below (and comment out the block above)
# if you want to run a fresh hourly-specific tuning in the future.
# ==========================================

# # Reload df_long from disk — build_product_sequences needs it to
# # build the hero product sequences for Optuna tuning.
# df_long = pd.read_parquet(DFLONG_PATH, engine='fastparquet')
#
# OPTUNA_TRIALS_PER_CLUSTER = 10
#
# print(f"Running Optuna on {N_CLUSTERS} hero products x {OPTUNA_TRIALS_PER_CLUSTER} trials each")
# print(f"Then picking the best params for ONE global hourly model\n")
#
# # --- Build hero-only training subsets ---
# cluster_subsets = {}
# for cid in range(N_CLUSTERS):
#     hero = CLUSTER_HEROES[cid]
#     if hero not in product_columns:
#         print(f"  Cluster {cid}: hero {hero} not in data, skipping")
#         continue
#
#     result = build_product_sequences(hero, SEQUENCE_LENGTH)
#     if result is None:
#         print(f"  Cluster {cid}: hero {hero} has insufficient data, skipping")
#         continue
#
#     if len(result['X_train']) > 0:
#         cluster_subsets[cid] = {
#             'X_train': result['X_train'],
#             'y_train': result['y_train'],
#             'X_val':   result['X_val'],
#             'y_val':   result['y_val']
#         }
#         print(f"  Cluster {cid} hero={hero}: {len(result['X_train'])} train, "
#               f"{len(result['X_val'])} val sequences")
#
# # --- Run Optuna per cluster (hero only) ---
# cluster_results = {}
#
# for cid in sorted(cluster_subsets.keys()):
#     cd = cluster_subsets[cid]
#
#     print(f"\n{'='*50}")
#     print(f"  Tuning Cluster {cid} — hero only: {CLUSTER_HEROES[cid]}")
#     print(f"  ({cd['X_train'].shape[0]} train sequences)")
#     print(f"{'='*50}")
#
#     def make_objective(Xt, yt, Xv, yv):
#         _Xt, _yt, _Xv, _yv = Xt, yt, Xv, yv
#         def objective(trial):
#             tf.keras.backend.clear_session()
#             batch_size    = trial.suggest_categorical('batch_size', [32, 64, 128])
#             lstm_1_units  = trial.suggest_int('lstm_1_units', 64, 256, step=32)
#             lstm_2_units  = trial.suggest_int('lstm_2_units', 32, 128, step=16)
#             dropout_rate  = trial.suggest_float('dropout_rate', 0.1, 0.4)
#             learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
#             l2_reg        = trial.suggest_float('l2_reg', 1e-5, 1e-2, log=True)
#             dense_units   = trial.suggest_categorical('dense_units', [32, 64])
#             model = Sequential([
#                 Input(shape=(_Xt.shape[1], _Xt.shape[2])),
#                 LSTM(lstm_1_units, activation='tanh', return_sequences=True),
#                 Dropout(dropout_rate),
#                 LSTM(lstm_2_units, activation='tanh'),
#                 Dropout(dropout_rate),
#                 Dense(dense_units, activation='relu', kernel_regularizer=l2(l2_reg)),
#                 Dense(1)
#             ])
#             model.compile(optimizer=Adam(learning_rate=learning_rate), loss=Huber())
#             train_ds, train_steps = make_tf_dataset(_Xt, _yt, batch_size, shuffle=True)
#             val_ds, val_steps     = make_tf_dataset(_Xv, _yv, batch_size)
#             model.fit(train_ds, validation_data=val_ds,
#                       epochs=100, steps_per_epoch=train_steps, validation_steps=val_steps,
#                       callbacks=[
#                           EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
#                           ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=0)
#                       ], verbose=0)
#             val_preds  = target_scaler.inverse_transform(model.predict(val_ds, steps=val_steps, verbose=0)).flatten()
#             val_actual = target_scaler.inverse_transform(_yv.reshape(-1, 1)).flatten()
#             return np.sqrt(mean_squared_error(val_actual, val_preds))
#         return objective
#
#     obj   = make_objective(cd['X_train'], cd['y_train'], cd['X_val'], cd['y_val'])
#     study = optuna.create_study(direction='minimize')
#     study.optimize(obj, n_trials=OPTUNA_TRIALS_PER_CLUSTER)
#     cluster_results[cid] = (study.best_params, study.best_value)
#     print(f"  Best RMSE: {study.best_value:.4f}")
#     del study, obj
#     tf.keras.backend.clear_session()
#     gc.collect()
#
# # --- Evaluate each cluster's best params on FULL validation set ---
# print(f"\n{'='*50}")
# print(f"  EVALUATING HERO PARAMS ON FULL VALIDATION SET")
# print(f"{'='*50}")
#
# best_global_rmse = float('inf')
# best_params      = None
# best_cluster_id  = None
#
# for cid, (params, hero_rmse) in cluster_results.items():
#     tf.keras.backend.clear_session()
#     model = Sequential([
#         Input(shape=(X_train_global.shape[1], X_train_global.shape[2])),
#         LSTM(params['lstm_1_units'], activation='tanh', return_sequences=True),
#         Dropout(params['dropout_rate']),
#         LSTM(params['lstm_2_units'], activation='tanh'),
#         Dropout(params['dropout_rate']),
#         Dense(params['dense_units'], activation='relu', kernel_regularizer=l2(params['l2_reg'])),
#         Dense(1)
#     ])
#     model.compile(optimizer=Adam(learning_rate=params['learning_rate']), loss=Huber())
#     train_ds, train_steps = make_tf_dataset(X_train_global, y_train_global, params['batch_size'], shuffle=True)
#     val_ds, val_steps     = make_tf_dataset(X_val_global,   y_val_global,   params['batch_size'])
#     model.fit(train_ds, validation_data=val_ds,
#               epochs=100, steps_per_epoch=train_steps, validation_steps=val_steps,
#               callbacks=[
#                   EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
#                   ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=0)
#               ], verbose=0)
#     val_preds  = target_scaler.inverse_transform(model.predict(val_ds, steps=val_steps, verbose=0)).flatten()
#     val_actual = target_scaler.inverse_transform(y_val_global.reshape(-1, 1)).flatten()
#     global_rmse = np.sqrt(mean_squared_error(val_actual, val_preds))
#
#     print(f"  Cluster {cid} ({CLUSTER_HEROES[cid]}) params -> global val RMSE: {global_rmse:.4f} "
#           f"(hero val RMSE was {hero_rmse:.4f})")
#
#     del model, train_ds, val_ds, train_steps, val_steps
#     tf.keras.backend.clear_session()
#     gc.collect()
#
#     if global_rmse < best_global_rmse:
#         best_global_rmse = global_rmse
#         best_params      = params
#         best_cluster_id  = cid
#
# print(f"\n  WINNER: Cluster {best_cluster_id} ({CLUSTER_HEROES[best_cluster_id]}) params")
# print(f"  Global Validation RMSE: {best_global_rmse:.4f}")
# print(f"  Params:")
# for key, value in best_params.items():
#     print(f"    {key}: {value}")
#
# del cluster_subsets, cluster_results, df_long
# gc.collect()


Skipping Optuna — params transferred from daily model (Cluster 0 / Bacon Waffle winner):
  batch_size: 32
  lstm_1_units: 160
  lstm_2_units: 96
  dropout_rate: 0.2618488090568859
  learning_rate: 0.00046818749855735817
  l2_reg: 1.005276167736066e-05
  dense_units: 64


In [6]:
# ==========================================
# 5. TRAIN FINAL GLOBAL MODEL
# ==========================================
print("Training final global hourly LSTM model with best parameters...")

tf.keras.backend.clear_session()

global_model = Sequential([
    Input(shape=(SEQUENCE_LENGTH, len(feature_cols))),
    LSTM(best_params.get('lstm_1_units', 128), activation='tanh', return_sequences=True),
    Dropout(best_params.get('dropout_rate', 0.2)),
    LSTM(best_params.get('lstm_2_units', 64), activation='tanh'),
    Dropout(best_params.get('dropout_rate', 0.2)),
    Dense(best_params.get('dense_units', 64), activation='relu',
          kernel_regularizer=l2(best_params.get('l2_reg', 0.001))),
    Dense(1)
])

global_model.compile(
    optimizer=Adam(learning_rate=best_params.get('learning_rate', 0.001)),
    loss=Huber()
)

batch_size = best_params.get('batch_size', 64)
train_ds, train_steps = make_tf_dataset(X_train_global, y_train_global, batch_size, shuffle=True)
val_ds, val_steps = make_tf_dataset(X_val_global, y_val_global, batch_size)

global_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    steps_per_epoch=train_steps,
    validation_steps=val_steps,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=0)
    ],
    verbose=1
)

global_model.summary()

# Training is done — free the memmap references and datasets.
# The files stay on disk but the OS can reclaim any cached pages.
# From here I only need global_model + product_test_data for forecasting.
del X_train_global, y_train_global, X_val_global, y_val_global, train_ds, val_ds, train_steps, val_steps
gc.collect()


Training final global hourly LSTM model with best parameters...


I0000 00:00:1776586073.852463    2826 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9534 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080 Ti, pci bus id: 0000:2d:00.0, compute capability: 8.6


Epoch 1/10
84603/84603 ━━━━━━━━━━━━━━━━━━━━ 1518s 18ms/step - loss: 1.8952e-04 - val_loss: 7.7493e-05 - learning_rate: 4.6819e-04
Epoch 2/10
84603/84603 ━━━━━━━━━━━━━━━━━━━━ 1536s 18ms/step - loss: 7.5346e-05 - val_loss: 8.3612e-05 - learning_rate: 4.6819e-04
Epoch 3/10
84603/84603 ━━━━━━━━━━━━━━━━━━━━ 1517s 18ms/step - loss: 6.9873e-05 - val_loss: 7.3040e-05 - learning_rate: 4.6819e-04
Epoch 4/10
84603/84603 ━━━━━━━━━━━━━━━━━━━━ 1548s 18ms/step - loss: 6.6783e-05 - val_loss: 7.8750e-05 - learning_rate: 4.6819e-04
Epoch 5/10
84603/84603 ━━━━━━━━━━━━━━━━━━━━ 1518s 18ms/step - loss: 6.5397e-05 - val_loss: 7.2583e-05 - learning_rate: 4.6819e-04
Epoch 6/10
84603/84603 ━━━━━━━━━━━━━━━━━━━━ 1531s 18ms/step - loss: 6.3876e-05 - val_loss: 7.3394e-05 - learning_rate: 4.6819e-04
Epoch 7/10
84603/84603 ━━━━━━━━━━━━━━━━━━━━ 1514s 18ms/step - loss: 6.1437e-05 - val_loss: 6.9926e-05 - learning_rate: 2.3409e-04
Epoch 8/10
84603/84603 ━━━━━━━━━━━━━━━━━━━━ 1535s 18ms/step - loss: 6.0255e-05 - val_loss:

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 63, 160)        │       124,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 63, 160)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 96)             │        98,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 96)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 687,365 (2.62 MB)

 Trainable params: 229,121 (895.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 458,244 (1.75 MB)

1508

In [7]:
# ==========================================
# 6. RECURSIVE FORECAST, ROLL UP & PER-PRODUCT METRICS (LEAK-FREE)
# ==========================================
# Fixed version: lag features are recomputed on the fly from PREDICTED
# values, not from pre-built X_test. This avoids data leakage where the
# model would otherwise see real future sales through the lag columns.
#
# Hourly lags: lag_1h, lag_9h, lag_63h, rolling_9h_mean, rolling_9h_std
# We maintain a sales_history buffer and recompute these at each step.

# Reload df_long from the parquet file I saved in Cell 3.
# This is needed because recursive_forecast_no_leak() reads the
# training/val sales history to seed the lag features.
df_long = pd.read_parquet(DFLONG_PATH, engine='fastparquet')

print("Running recursive forecast for ALL products (leak-free lags)...")
print("(Lag features recomputed from predictions — no data leakage)\n")

# Work out which feature indices correspond to lags
# feature_cols = base_feature_cols + ['Product_ID'] + lag_features
lag_feature_names = ['sales_lag_1h', 'sales_lag_9h', 'sales_lag_63h', 'sales_rolling_9h_mean', 'sales_rolling_9h_std']
n_features = len(feature_cols)
n_lags = len(lag_feature_names)
lag_start_idx = n_features - n_lags

# Extract scaler parameters for lag columns so we can manually scale
lag_scaler_min = feature_scaler.data_min_[lag_start_idx:]
lag_scaler_scale = feature_scaler.data_range_[lag_start_idx:]
lag_scaler_scale[lag_scaler_scale == 0] = 1.0  # avoid division by zero


def scale_lags(lag_values):
    """Scale raw lag values using the fitted feature_scaler's parameters."""
    return (lag_values - lag_scaler_min) / lag_scaler_scale


def recursive_forecast_no_leak(model, X_test, seq_len, tgt_scaler, y_test, product_name):
    """
    Recursive forecast that recomputes hourly lag features from predictions.
    
    Instead of trusting the pre-built lag columns in X_test (which use
    real future sales), we maintain our own sales history and recompute
    lag_1h, lag_9h, lag_63h, rolling_9h_mean, and rolling_9h_std at each step.
    """
    # Get the last SEQUENCE_LENGTH hours of real sales BEFORE the test period
    # We need at least 63 hours of history to compute lag_63h
    product_train_val = df_long[
        (df_long['Product_Name'] == product_name) & (df_long['Date'] <= val_end)
    ].sort_values('Date')['Sales'].values
    
    # Take enough history to seed all lags (at least 63 for lag_63h)
    seed_len = max(seq_len, 63)
    sales_history = list(product_train_val[-seed_len:])
    
    current_seq = X_test[0].copy()
    preds_scaled = []
    
    for i in range(len(X_test)):
        # 1. Predict one hour
        pred_scaled = model.predict(current_seq.reshape(1, seq_len, -1), verbose=0)[0, 0]
        preds_scaled.append(pred_scaled)
        
        # Convert prediction back to real units and add to history
        pred_real = tgt_scaler.inverse_transform([[pred_scaled]])[0, 0]
        pred_real = max(pred_real, 0)
        sales_history.append(pred_real)
        
        if i + 1 < len(X_test):
            # 2. Get next hour's NON-lag features from X_test (weather, time, product_id)
            next_feats = X_test[i + 1][-1, :].copy()
            
            # 3. Recompute lag features from sales_history (uses predictions)
            h = sales_history
            lag_1h = h[-1]
            lag_9h = h[-9] if len(h) >= 9 else h[0]
            lag_63h = h[-63] if len(h) >= 63 else h[0]
            
            # Rolling 9h mean/std on shift(1) — i.e. the last 9 values ending at h[-1]
            rolling_window = h[-9:] if len(h) >= 9 else h[:]
            rolling_9h_mean = np.mean(rolling_window)
            rolling_9h_std = np.std(rolling_window, ddof=0) if len(rolling_window) > 1 else 0.0
            
            raw_lags = np.array([lag_1h, lag_9h, lag_63h, rolling_9h_mean, rolling_9h_std])
            scaled_lags = scale_lags(raw_lags)
            
            # 4. Overwrite the lag columns with recomputed values
            next_feats[lag_start_idx:] = scaled_lags
            
            # 5. Slide the window forward
            current_seq = np.roll(current_seq, -1, axis=0)
            current_seq[-1, :] = next_feats
    
    preds_real = tgt_scaler.inverse_transform(np.array(preds_scaled).reshape(-1, 1)).flatten()
    preds_real = np.maximum(preds_real, 0)
    actual_real = tgt_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
    return preds_real, actual_real


def calculate_horizon_metrics(df_slice, in_sample_naive_mae):
    """Calculate all metrics — identical to per-product version."""
    if len(df_slice) == 0:
        return [0] * 7
    y_true = df_slice['Sales'].values
    y_pred = df_slice['LSTM_Prediction'].values
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    numerator = np.abs(y_pred - y_true)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    smape = np.mean(numerator / (denominator + 1e-8))
    mase = mae / (in_sample_naive_mae + 1e-8)
    forecast_bias = np.mean(y_pred - y_true)
    return [mae, mse, rmse, mape, smape, mase, forecast_bias]


# ── helper: formatted product scorecard ──────────────────────
def print_product_scorecard(product_name, product_idx, total_products,
                            metrics_df, daily_rollup):
    """Print a clean, aligned scorecard for one product."""
    w = 72  # total box width

    # ── header ──
    print(f"\n{'┌' + '─' * (w - 2) + '┐'}")
    title = f"Product {product_idx}/{total_products}: {product_name}"
    print(f"│{title:^{w - 2}}│")
    print(f"{'├' + '─' * (w - 2) + '┤'}")

    # ── metrics table ──
    horizons = list(metrics_df.columns[1:])
    metric_names = metrics_df['Metric'].tolist()

    # column widths
    lbl_w = max(len(m) for m in metric_names) + 2
    col_w = max(len(h) for h in horizons) + 2
    row_fmt = f"│ {{:<{lbl_w}}}│" + "│".join([f"{{:>{col_w}}}"] * len(horizons)) + f"│"

    # header row
    print(row_fmt.format("Metric", *horizons))
    print(f"│{'─' * (lbl_w + 1)}┼" + "┼".join(["─" * col_w] * len(horizons)) + f"│")

    for _, row in metrics_df.iterrows():
        vals = [str(row[h]) for h in horizons]
        print(row_fmt.format(row['Metric'], *vals))

    print(f"{'├' + '─' * (w - 2) + '┤'}")

    # ── reality check (first 10 days) ──
    rc = daily_rollup[['Date_Only', 'Sales', 'LSTM_Prediction_Rounded',
                        'Mistake_Amount']].head(10).copy()
    rc.columns = ['Date', 'Actual', 'Predicted', 'Error']
    rc['Date'] = rc['Date'].astype(str)

    sub_title = "Reality Check — Daily Totals (first 10 days)"
    print(f"│{sub_title:^{w - 2}}│")
    print(f"│{'─' * (w - 2)}│")

    d_w, a_w, p_w, e_w = 12, 8, 10, 7
    rc_fmt = f"│  {{:<{d_w}}} {{:>{a_w}}} {{:>{p_w}}} {{:>{e_w}}}  │"

    print(rc_fmt.format("Date", "Actual", "Predicted", "Error"))
    print(f"│  {'─' * d_w} {'─' * a_w} {'─' * p_w} {'─' * e_w}  │")
    for _, r in rc.iterrows():
        print(rc_fmt.format(
            str(r['Date']),
            str(int(r['Actual'])),
            str(int(r['Predicted'])),
            str(int(r['Error']))
        ))

    print(f"{'└' + '─' * (w - 2) + '┘'}")


# --- Run forecast, rollup, and metrics for every product ---
all_results = {}
all_test_dfs = {}

# Only forecast the 74 target products
products_to_predict = [p for p in products_included if p in PRODUCTS_TO_FORECAST]

for product_idx, product_name in enumerate(products_to_predict, start=1):
    test_info = product_test_data[product_name]
    X_test = test_info['X_test']
    y_test = test_info['y_test']
    test_dates = test_info['test_dates']
    
    if len(X_test) == 0:
        print(f"\n  SKIPPING {product_name}: no test data")
        continue
    
    # --- Recursive hourly forecast with leak-free lags ---
    preds_real, actual_real = recursive_forecast_no_leak(
        global_model, X_test, SEQUENCE_LENGTH, target_scaler, y_test, product_name
    )
    
    # --- Roll up hourly to daily (same as per-product version) ---
    test_df = pd.DataFrame({
        'Date': test_dates,
        'Hourly_Sales': actual_real,
        'Hourly_LSTM_Prediction': preds_real
    })
    test_df['Date_Only'] = test_df['Date'].dt.date
    
    daily_rollup = test_df.groupby('Date_Only')[['Hourly_Sales', 'Hourly_LSTM_Prediction']].sum().reset_index()
    daily_rollup = daily_rollup.rename(columns={
        'Hourly_Sales': 'Sales',
        'Hourly_LSTM_Prediction': 'LSTM_Prediction'
    })
    
    # Using the naive MAE I pre-computed in Cell 3
    in_sample_naive_mae = product_naive_mae.get(product_name, 1.0)
    
    # --- Calculate metrics on daily rollup ---
    t1 = pd.to_datetime('2025-11-02').date()
    t7 = pd.to_datetime('2025-11-08').date()
    t30 = pd.to_datetime('2025-11-30').date()
    
    df_1d = daily_rollup[daily_rollup['Date_Only'] == t1]
    df_1w = daily_rollup[(daily_rollup['Date_Only'] >= t1) & (daily_rollup['Date_Only'] <= t7)]
    df_1m = daily_rollup[(daily_rollup['Date_Only'] >= t1) & (daily_rollup['Date_Only'] <= t30)]
    
    metrics_df = pd.DataFrame({
        'Metric': ['MAE', 'MSE', 'RMSE', 'MAPE', 'SMAPE', 'MASE', 'Bias'],
        '1-Day (Nov 2)': calculate_horizon_metrics(df_1d, in_sample_naive_mae),
        '1-Week (Nov 2-8)': calculate_horizon_metrics(df_1w, in_sample_naive_mae),
        '1-Month (Nov 2-30)': calculate_horizon_metrics(df_1m, in_sample_naive_mae)
    })
    
    for col in metrics_df.columns[1:]:
        metrics_df[col] = metrics_df[col].apply(lambda x: f"{x:.4f}")
    
    daily_rollup['LSTM_Prediction_Rounded'] = daily_rollup['LSTM_Prediction'].round().astype(int)
    daily_rollup['Mistake_Amount'] = (daily_rollup['Sales'] - daily_rollup['LSTM_Prediction_Rounded']).abs()
    
    # Print the clean scorecard
    print_product_scorecard(product_name, product_idx, len(products_to_predict),
                            metrics_df, daily_rollup)
    
    all_results[product_name] = metrics_df
    all_test_dfs[product_name] = daily_rollup

    # Free this product's test arrays before moving to the next one
    del test_info, X_test, y_test, test_dates, preds_real, actual_real, test_df
    gc.collect()

print(f"\n{'┌' + '─' * 70 + '┐'}")
done_msg = f"ALL PRODUCTS COMPLETE ({len(all_results)}/{len(products_to_predict)})"
print(f"│{done_msg:^70}│")
print(f"{'└' + '─' * 70 + '┘'}")


# All products forecasted — free df_long again, only need all_results
# and all_test_dfs (daily rollups, much smaller) from here
del df_long
gc.collect()


Running recursive forecast for ALL products (leak-free lags)...
(Lag features recomputed from predictions — no data leakage)


┌──────────────────────────────────────────────────────────────────────┐
│                    Product 1/64: Avo & Hal Muffin                    │
├──────────────────────────────────────────────────────────────────────┤
│ Metric │       1-Day (Nov 2)│    1-Week (Nov 2-8)│  1-Month (Nov 2-30)│
│────────┼────────────────────┼────────────────────┼────────────────────│
│ MAE    │              0.1704│              0.1704│              0.1704│
│ MSE    │              0.0290│              0.0290│              0.0290│
│ RMSE   │              0.1704│              0.1704│              0.1704│
│ MAPE   │767589642928128.0000│767589642928128.0000│767589710036992.0000│
│ SMAPE  │              2.0000│              2.0000│              2.0000│
│ MASE   │              0.3694│              0.3694│              0.3694│
│ Bias   │              0.1704│              0.1704│          

0

In [8]:
# ==========================================
# 7. SUMMARY TABLE ACROSS ALL PRODUCTS
# ==========================================

summary_rows = []
for product_name, metrics_df in all_results.items():
    mae_val = metrics_df.loc[metrics_df['Metric'] == 'MAE', '1-Month (Nov 2-30)'].values[0]
    rmse_val = metrics_df.loc[metrics_df['Metric'] == 'RMSE', '1-Month (Nov 2-30)'].values[0]
    mase_val = metrics_df.loc[metrics_df['Metric'] == 'MASE', '1-Month (Nov 2-30)'].values[0]
    bias_val = metrics_df.loc[metrics_df['Metric'] == 'Bias', '1-Month (Nov 2-30)'].values[0]
    summary_rows.append({'Product': product_name, 'MAE': mae_val, 'RMSE': rmse_val, 'MASE': mase_val, 'Bias': bias_val})

summary_df = pd.DataFrame(summary_rows)

# ── Pretty-print the summary table ──
w = 82
print(f"\n{'┌' + '─' * (w - 2) + '┐'}")
print(f"│{'SUMMARY: 1-Month Metrics — Global Hourly LSTM (All Products)':^{w - 2}}│")
print(f"{'├' + '─' * (w - 2) + '┤'}")

p_w = 35  # product name width
m_w = 9   # metric column width
hdr = f"│ {'Product':<{p_w}} │ {'MAE':>{m_w}} │ {'RMSE':>{m_w}} │ {'MASE':>{m_w}} │ {'Bias':>{m_w}} │"
print(hdr)
print(f"│{'─' * (p_w + 2)}┼{'─' * (m_w + 2)}┼{'─' * (m_w + 2)}┼{'─' * (m_w + 2)}┼{'─' * (m_w + 2)}│")

for _, row in summary_df.iterrows():
    name = row['Product'][:p_w]
    print(f"│ {name:<{p_w}} │ {str(row['MAE']):>{m_w}} │ {str(row['RMSE']):>{m_w}} │ {str(row['MASE']):>{m_w}} │ {str(row['Bias']):>{m_w}} │")

# ── Averages row ──
avg_mae  = summary_df['MAE'].astype(float).mean()
avg_rmse = summary_df['RMSE'].astype(float).mean()
avg_mase = summary_df['MASE'].astype(float).mean()
avg_bias = summary_df['Bias'].astype(float).mean()
print(f"│{'─' * (p_w + 2)}┼{'─' * (m_w + 2)}┼{'─' * (m_w + 2)}┼{'─' * (m_w + 2)}┼{'─' * (m_w + 2)}│")
print(f"│ {'AVERAGE':<{p_w}} │ {avg_mae:>{m_w}.4f} │ {avg_rmse:>{m_w}.4f} │ {avg_mase:>{m_w}.4f} │ {avg_bias:>{m_w}.4f} │")
print(f"{'└' + '─' * (w - 2) + '┘'}")

os.makedirs('../results', exist_ok=True)
summary_df.to_csv('../results/lstm_global_hourly_all_products_summary.csv', index=False)
print("\nSaved to ../results/lstm_global_hourly_all_products_summary.csv")



┌────────────────────────────────────────────────────────────────────────────────┐
│          SUMMARY: 1-Month Metrics — Global Hourly LSTM (All Products)          │
├────────────────────────────────────────────────────────────────────────────────┤
│ Product                             │       MAE │      RMSE │      MASE │      Bias │
│─────────────────────────────────────┼───────────┼───────────┼───────────┼───────────│
│ Avo & Hal Muffin                    │    0.1704 │    0.1704 │    0.3694 │    0.1704 │
│ Avo, Egg & Bacon                    │    2.8357 │    3.5151 │    5.7688 │   -2.6380 │
│ Avo, Feta & Tom                     │    1.9479 │    2.5467 │    4.1230 │   -1.9114 │
│ Avocado on Toast                    │    1.5816 │    2.0797 │    0.7529 │   -1.5208 │
│ Bacon                               │    3.5140 │    4.3365 │    0.8814 │   -3.1535 │
│ Bacon Bap                           │    5.5258 │    6.7005 │    1.1170 │   -4.2266 │
│ Bacon Egg Brioch                    │    1.2

In [9]:
# Free the trained model — only need the prediction results from here
try:
    del global_model
    tf.keras.backend.clear_session()
    gc.collect()
except NameError:
    pass

# ==========================================
# 8. SAVE RESULTS
# ==========================================
import sqlite3
from datetime import datetime

os.makedirs('../results', exist_ok=True)

# --- Generate a unique run ID ---
prediction_timestamp = datetime.now()
run_id = prediction_timestamp.strftime('%Y%m%d_%H%M') + '_LSTM_Global_Hourly'

print(f"Saving results for run: {run_id}")

# --- 1. Save CSVs ---
summary_df.to_csv('../results/lstm_global_hourly_per_product_results.csv', index=False)

all_preds_rows = []
for product_name, daily_rollup in all_test_dfs.items():
    for _, row in daily_rollup.iterrows():
        all_preds_rows.append({
            'Date': row['Date_Only'],
            'Product_Name': product_name,
            'Sales': row['Sales'],
            'LSTM_Prediction': row['LSTM_Prediction']
        })
all_preds_df = pd.DataFrame(all_preds_rows)
all_preds_df.to_csv('../results/lstm_global_hourly_all_predictions.csv', index=False)
print("  CSVs saved to ../results/")

# --- 2. SQLite Storage Framework ---
database_path = '../results/model_tracking.db'
db_connection = sqlite3.connect(database_path)

with db_connection:
    db_connection.execute("""
        CREATE TABLE IF NOT EXISTS predictions_log (
            run_id TEXT, model_type TEXT, target_date TEXT,
            prediction_made_on TEXT, product_name TEXT,
            actual_sales REAL, predicted_sales REAL
        )
    """)
    db_connection.execute("""
        CREATE TABLE IF NOT EXISTS metrics_summary (
            run_id TEXT, model_type TEXT, product_name TEXT,
            evaluation_horizon TEXT, WAPE REAL, MASE REAL, MAE REAL, Bias REAL
        )
    """)
    
    # --- Populate predictions_log ---
    predictions_for_db = all_preds_df.rename(columns={
        'Date': 'target_date', 'Product_Name': 'product_name',
        'Sales': 'actual_sales', 'LSTM_Prediction': 'predicted_sales'
    })
    predictions_for_db['run_id'] = run_id
    predictions_for_db['model_type'] = 'LSTM_Global_Hourly'
    predictions_for_db['prediction_made_on'] = str(prediction_timestamp)
    predictions_for_db['target_date'] = predictions_for_db['target_date'].astype(str)
    predictions_for_db.to_sql('predictions_log', db_connection, if_exists='append', index=False)
    print(f"  Logged {len(predictions_for_db)} predictions to predictions_log")
    
    # --- Populate metrics_summary ---
    t1 = pd.to_datetime('2025-11-02').date()
    t7 = pd.to_datetime('2025-11-08').date()
    t30 = pd.to_datetime('2025-11-30').date()
    
    metrics_rows = []
    for product_name, daily_rollup in all_test_dfs.items():
        for horizon_label, start, end in [('1-Day', t1, t1), ('1-Week', t1, t7), ('1-Month', t1, t30)]:
            if horizon_label == '1-Day':
                horizon_df = daily_rollup[daily_rollup['Date_Only'] == start]
            else:
                horizon_df = daily_rollup[(daily_rollup['Date_Only'] >= start) & (daily_rollup['Date_Only'] <= end)]
            
            if len(horizon_df) > 0:
                actual = horizon_df['Sales'].values
                predicted = horizon_df['LSTM_Prediction'].values
                
                total_abs_err = np.sum(np.abs(actual - predicted))
                total_actual = np.sum(np.abs(actual))
                wape = total_abs_err / total_actual if total_actual > 0 else np.nan
                mae = mean_absolute_error(actual, predicted)
                
                naive_mae = product_naive_mae.get(product_name, 1.0)
                mase = mae / (naive_mae + 1e-8)
                bias = np.mean(predicted - actual)
                
                metrics_rows.append({
                    'run_id': run_id, 'model_type': 'LSTM_Global_Hourly',
                    'product_name': product_name, 'evaluation_horizon': horizon_label,
                    'WAPE': wape, 'MASE': mase, 'MAE': mae, 'Bias': bias
                })
    
    # ALL_PRODUCTS aggregate rows
    combined_rollup = pd.concat(all_test_dfs.values(), ignore_index=True)
    
    global_naive_mae = np.mean([product_naive_mae[p] for p in products_included if p in product_naive_mae])
    
    for horizon_label, start, end in [('1-Day', t1, t1), ('1-Week', t1, t7), ('1-Month', t1, t30)]:
        if horizon_label == '1-Day':
            horizon_df = combined_rollup[combined_rollup['Date_Only'] == start]
        else:
            horizon_df = combined_rollup[(combined_rollup['Date_Only'] >= start) & (combined_rollup['Date_Only'] <= end)]
        
        if len(horizon_df) > 0:
            actual = horizon_df['Sales'].values
            predicted = horizon_df['LSTM_Prediction'].values
            total_abs_err = np.sum(np.abs(actual - predicted))
            total_actual = np.sum(np.abs(actual))
            
            metrics_rows.append({
                'run_id': run_id, 'model_type': 'LSTM_Global_Hourly',
                'product_name': 'ALL_PRODUCTS', 'evaluation_horizon': horizon_label,
                'WAPE': total_abs_err / total_actual if total_actual > 0 else np.nan,
                'MASE': mean_absolute_error(actual, predicted) / (global_naive_mae + 1e-8),
                'MAE': mean_absolute_error(actual, predicted),
                'Bias': np.mean(predicted - actual)
            })
    
    if metrics_rows:
        metrics_summary_df = pd.DataFrame(metrics_rows)
        metrics_summary_df.to_sql('metrics_summary', db_connection, if_exists='append', index=False)
        print(f"  Logged {len(metrics_summary_df)} metric rows to metrics_summary")

db_connection.close()
print(f"  SQLite database saved to {database_path}")
print(f"  Run ID: {run_id}")
print("\nDone!")

# Clean up the temporary memmap files from disk — don't need them anymore
import shutil
try:
    shutil.rmtree(MEMMAP_DIR)
    print(f"  Cleaned up temp dir: {MEMMAP_DIR}")
except:
    pass

Saving results for run: 20260419_1339_LSTM_Global_Hourly
  CSVs saved to ../results/
  Logged 1856 predictions to predictions_log
  Logged 195 metric rows to metrics_summary
  SQLite database saved to ../results/model_tracking.db
  Run ID: 20260419_1339_LSTM_Global_Hourly

Done!
  Cleaned up temp dir: /tmp/lstm_memmap_0om535cs
